# Unit 7 — Simulation & Ad Hoc

A contest robot receives a long list of commands, but walls, turns, and earlier commands change what later commands do.
Instead of guessing a shortcut, we can follow the rules exactly.
This unit builds a simulation: model the changing state, apply every step in order, and check the edges that can stop or redirect a step.

## Lesson 1 — 1. Model the State

State is the information that can change and that the next rule needs.
A counter may need one integer, a row of tiles may need a list, and a board may need a grid plus the robot's row and column.
Before writing the loop, name each state variable and its starting value.
Do not save details that no later step or final answer uses.

## A Capped Battery State

This battery has a capacity, a current charge, and one change for each tick.
After every change, charge above the capacity becomes the capacity and charge below zero becomes zero.
The clamping after each tick matters because the next tick begins from the updated charge.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    capacity = int(tokens[0])
    charge = int(tokens[1])
    ticks = int(tokens[2])
    for tick in range(ticks):
        change = int(tokens[tick + 3])
        charge = charge + change
        if charge > capacity:
            charge = capacity
        elif charge < 0:
            charge = 0
    return str(charge)

assert solve("10 4 6 9 -3 -20 5 8 -2") == "8"

## 2. Apply Every Rule in Order

Read the steps in input order and update the state once for each step.
Within one step, keep the problem's rule order too: calculate the proposed change, decide whether it is allowed, and only then store the new state.
A later step must see the state left by the earlier steps.
A loop over all `ticks` includes the last tick; stopping at `ticks - 1` silently drops the step that may decide the answer.

## Lesson 2 — 3. Watch Edges and Termination

A grid move has two stages: propose a neighboring cell, then check that its row and column are inside the grid before reading that cell.
If the proposed cell is outside the grid or is a wall, this robot stays put for that command.
The grid, current row, and current column are all part of the model.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    rows = int(tokens[0])
    columns = int(tokens[1])
    robot_row = int(tokens[2])
    robot_column = int(tokens[3])
    token_position = 4
    grid = []
    for row in range(rows):
        current_row = []
        for column in range(columns):
            current_row.append(int(tokens[token_position]))
            token_position = token_position + 1
        grid.append(current_row)
    commands = tokens[token_position]

    for command in commands:
        next_row = robot_row
        next_column = robot_column
        if command == "U":
            next_row = next_row - 1
        elif command == "D":
            next_row = next_row + 1
        elif command == "L":
            next_column = next_column - 1
        elif command == "R":
            next_column = next_column + 1
        if next_row >= 0 and next_row < rows and next_column >= 0 and next_column < columns:
            if grid[next_row][next_column] != -1:
                robot_row = next_row
                robot_column = next_column
    return str(grid[robot_row][robot_column])

assert solve("3 4 1 0 5 2 3 4 6 -1 8 6 7 1 4 9 DDRRRU") == "6"

## Make Every Loop Provably Stop

A `for` loop over a fixed number of commands is automatically bounded.
For a `while` loop, identify the progress that brings it toward its stopping condition.
In the event walk below, every jump is at least `1`, so the index increases on every pass and eventually reaches or passes the list length.
Check the stopping condition before indexing; a final jump is allowed to land beyond the list.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    event_count = int(tokens[0])
    points = []
    jumps = []
    for event in range(event_count):
        points.append(int(tokens[1 + event * 2]))
        jumps.append(int(tokens[2 + event * 2]))

    score = 0
    event_index = 0
    while event_index < event_count:
        score = score + points[event_index]
        event_index = event_index + jumps[event_index]
    return str(score)

assert solve("7 5 2 100 1 3 1 4 3 100 1 100 1 9 2") == "21"

## A Simulation Checklist

First, write down the starting state.
Second, translate one step into code in the exact stated order and repeat it for every command, tick, or event.
Third, test the first and last steps, grid borders, blocked moves, and the condition that ends every loop.
A small hand trace is often the fastest way to catch an update made too early or a final step accidentally skipped.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))